# 6. Глобальный бустинг для прерывистого спроса

**Сквозной вопрос:** когда глобальная модель лучше специализированных методов (Croston/ADIDA).

В этом ноутбуке:
- Постановка: M5 CA_1, признаки для прерывистого спроса;
- Функции потерь: RMSE vs Poisson vs Tweedie;
- Сравнение с семейством Croston по классам;
- Контрпример: известные нули (`is_closed`) вместо Croston;
- Итоговая памятка выбора подхода.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostRegressor
from statsforecast import StatsForecast
from statsforecast.models import CrostonSBA, TSB, ADIDA

sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 4)

## 0. Данные: M5 CA_1

Те же 3049 товаров, H=28. Добавляем календарные признаки и события.

In [ ]:
# Загрузка
sales = pd.read_csv('../data/sales_train_validation.csv')
calendar = pd.read_csv('../data/calendar.csv')

sales_ca1 = sales[sales['store_id'] == 'CA_1'].copy()

id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_ca1.columns if c.startswith('d_')]

df_long = sales_ca1.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

# Добавляем календарь
df_long = df_long.merge(calendar[['d', 'date', 'wm_yr_wk', 'event_name_1', 'event_type_1',
                                   'snap_CA', 'weekday']], on='d', how='left')
df_long['date'] = pd.to_datetime(df_long['date'])
df_long = df_long.sort_values(['id', 'date']).reset_index(drop=True)

print(f"Товаров: {df_long['id'].nunique()}, период: {df_long['date'].min()} — {df_long['date'].max()}")

In [ ]:
# Классификация (из 10.B.1)
def classify_demand(series):
    nonzero_idx = np.where(series > 0)[0]

    if len(nonzero_idx) == 0:
        return {'adi': np.inf, 'cv2': np.inf, 'class': 'lumpy'}

    if len(nonzero_idx) == 1:
        adi = len(series)
        cv2 = 0
    else:
        intervals = np.diff(nonzero_idx)
        adi = intervals.mean() if len(intervals) > 0 else len(series)

        nonzero_vals = series[nonzero_idx]
        if nonzero_vals.mean() > 0:
            cv2 = (nonzero_vals.std() / nonzero_vals.mean()) ** 2
        else:
            cv2 = 0

    if adi < 1.32 and cv2 < 0.49:
        demand_class = 'smooth'
    elif adi >= 1.32 and cv2 < 0.49:
        demand_class = 'intermittent'
    elif adi < 1.32 and cv2 >= 0.49:
        demand_class = 'erratic'
    else:
        demand_class = 'lumpy'

    return {'adi': adi, 'cv2': cv2, 'class': demand_class}

In [ ]:
classifications = []
for item_id in df_long['id'].unique():
    item_sales = df_long[df_long['id'] == item_id]['sales'].values
    result = classify_demand(item_sales)
    result['id'] = item_id
    classifications.append(result)

class_df = pd.DataFrame(classifications)

print(f"\nРаспределение классов:\n{class_df['class'].value_counts()}")

## 1. Признаки для прерывистого спроса

Ключевые особенности:
- **Лаги ≥ H (28)** — иначе утечка информации;
- **Скользящие статистики** со сдвигом H;
- **Календарь:** день недели, месяц, события, SNAP;
- **Категориальные:** `item_id`, `dept_id`, `cat_id`.

In [ ]:
def make_features(df, H=28):
    """
    Создаём признаки для прогноза с горизонтом H.
    """
    df = df.copy()

    # Календарные
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)

    # Событийные (заполняем NaN)
    df['has_event'] = (~df['event_name_1'].isna()).astype(int)
    df['snap'] = df['snap_CA'].fillna(0).astype(int)

    # Лаги
    for lag in [H, H+7, H+14, H+21, H+28]:
        df[f'lag_{lag}'] = df.groupby('id')['sales'].shift(lag)

    # Скользящие окна (со сдвигом H)
    for window in [7, 14, 28]:
        df[f'rolling_mean_{window}'] = df.groupby('id')['sales'].shift(H).rolling(window, min_periods=1).mean().reset_index(0, drop=True)
        df[f'rolling_std_{window}'] = df.groupby('id')['sales'].shift(H).rolling(window, min_periods=1).std().reset_index(0, drop=True)

    # Доля нулей в последних H днях
    df['zero_fraction_H'] = df.groupby('id')['sales'].shift(H).rolling(H, min_periods=1).apply(lambda x: (x == 0).mean()).reset_index(0, drop=True)

    return df

In [ ]:
# Применяем
df_feats = make_features(df_long, H=28)

# Удаляем строки с NaN (начальные)
df_feats = df_feats.dropna()

print(f"После создания признаков: {len(df_feats)} строк")

In [ ]:
# Train/test split: последние H дней — тест
train_cutoff = df_feats['date'].max() - pd.Timedelta(days=28)

train_df = df_feats[df_feats['date'] <= train_cutoff].copy()
test_df = df_feats[df_feats['date'] > train_cutoff].copy()

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Train период: {train_df['date'].min()} — {train_df['date'].max()}")
print(f"Test период: {test_df['date'].min()} — {test_df['date'].max()}")

In [ ]:
# Признаки и таргет
feature_cols = [c for c in df_feats.columns if c.startswith(('lag_', 'rolling_', 'zero_', 'day_', 'month', 'week_', 'has_', 'snap'))]
cat_features = ['item_id', 'dept_id', 'cat_id']

X_train = train_df[feature_cols + cat_features]
y_train = train_df['sales']

X_test = test_df[feature_cols + cat_features]
y_test = test_df['sales']

# Добавляем класс спроса для стратификации
train_df = train_df.merge(class_df[['id', 'class']], on='id', how='left')
test_df = test_df.merge(class_df[['id', 'class']], on='id', how='left')

print(f"Признаков: {len(feature_cols)} численных + {len(cat_features)} категориальных")

## 2. Функции потерь: RMSE vs Poisson vs Tweedie

### RMSE (квадратичная)
- Стандартная функция для регрессии
- Чувствительна к выбросам
- Не учитывает неотрицательность и дискретность

### Poisson
- Для счётных данных (0, 1, 2, ...)
- E[y] = Var[y] = λ (равнодисперсность)
- **Проблема:** на практике variance > mean (overdispersion)

### Tweedie
- Обобщение: масса в нуле + непрерывное распределение для ненулевых
- `variance_power` ∈ [1, 2]:
  - 1.0 → Poisson
  - 1.5 → compound Poisson-Gamma (хорош для прерывистых данных)
  - 2.0 → Gamma

In [ ]:
# Обучаем три модели
models_dict = {}

# 1. RMSE
print("Обучение CatBoost RMSE...")
cb_rmse = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='RMSE',
    cat_features=cat_features,
    verbose=False,
    random_state=42
)
cb_rmse.fit(X_train, y_train)
models_dict['CatBoost RMSE'] = cb_rmse

In [ ]:
# 2. Poisson
print("Обучение CatBoost Poisson...")
cb_poisson = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Poisson',
    cat_features=cat_features,
    verbose=False,
    random_state=42
)
cb_poisson.fit(X_train, y_train)
models_dict['CatBoost Poisson'] = cb_poisson

In [ ]:
# 3. Tweedie
print("Обучение CatBoost Tweedie...")
cb_tweedie = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function='Tweedie:variance_power=1.5',
    cat_features=cat_features,
    verbose=False,
    random_state=42
)
cb_tweedie.fit(X_train, y_train)
models_dict['CatBoost Tweedie'] = cb_tweedie

print("Все модели обучены.")

## 3. Сравнение с семейством Croston

Baseline: CrostonSBA, TSB, ADIDA из statsforecast.

In [ ]:
# Подготовка для statsforecast
H = 28

# Train
train_sf = train_df[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})
test_sf = test_df[['id', 'date', 'sales']].rename(columns={'id': 'unique_id', 'date': 'ds', 'sales': 'y'})

# Croston модели
sf_models = [CrostonSBA(), TSB(alpha_d=0.1, alpha_p=0.1), ADIDA()]
sf = StatsForecast(models=sf_models, freq='D', n_jobs=1)

print("Обучение Croston моделей...")
sf.fit(train_sf)
forecast_sf = sf.predict(h=H)

In [ ]:
# Прогнозы CatBoost
for name, model in models_dict.items():
    preds = model.predict(X_test)
    preds = np.maximum(preds, 0)  # обрезаем отрицательные
    test_df[name] = preds

# Прогнозы Croston
forecast_sf_reset = forecast_sf.reset_index()
for model_name in ['CrostonSBA', 'TSB', 'ADIDA']:
    test_df = test_df.merge(
        forecast_sf_reset[['unique_id', model_name]],
        left_on='id',
        right_on='unique_id',
        how='left',
        suffixes=('', '_sf')
    )
    test_df.drop(columns=['unique_id'], inplace=True, errors='ignore')

## 4. Метрики по классам спроса

RMSE и bias по каждому классу (smooth / intermittent / erratic / lumpy).

In [ ]:
def compute_metrics_by_class(df, model_cols):
    """
    Вычисляем RMSE и bias по классам спроса.
    """
    results = []

    for model_name in model_cols:
        for cls in ['smooth', 'intermittent', 'erratic', 'lumpy', 'all']:
            if cls == 'all':
                subset = df
            else:
                subset = df[df['class'] == cls]

            if len(subset) == 0:
                continue

            y_true = subset['sales'].values
            y_pred = subset[model_name].values

            rmse = np.sqrt(((y_true - y_pred) ** 2).mean())
            bias = (y_true - y_pred).mean()

            results.append({
                'Модель': model_name,
                'Класс': cls,
                'RMSE': rmse,
                'Bias': bias,
                'Записей': len(subset)
            })

    return pd.DataFrame(results)

In [ ]:
model_cols = ['CatBoost RMSE', 'CatBoost Poisson', 'CatBoost Tweedie', 'CrostonSBA', 'TSB', 'ADIDA']
results_df = compute_metrics_by_class(test_df, model_cols)

In [ ]:
# Визуализация RMSE по классам
pivot = results_df[results_df['Класс'] != 'all'].pivot(index='Модель', columns='Класс', values='RMSE')

pivot.plot(kind='bar', figsize=(14, 6))
plt.ylabel('RMSE')
plt.title('Сравнение: CatBoost vs Croston по классам спроса')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Класс спроса')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("\nМетрики по классам спроса:\n")
print(results_df[results_df['Класс'] != 'all'][['Модель', 'Класс', 'RMSE', 'Bias']].to_string(index=False))

### Наблюдения

- **Smooth:** глобальная модель обычно лучше Croston (меньше шума).
- **Intermittent:** Tweedie close к CrostonSBA/TSB, Poisson немного хуже.
- **Erratic:** глобальная модель может сгладить выбросы.
- **Lumpy:** ADIDA лучше CatBoost RMSE, но Tweedie конкурентен.

## 5. Важности признаков

Какие признаки важны для Tweedie?

In [ ]:
importances = cb_tweedie.get_feature_importance()
feature_names = cb_tweedie.feature_names_

# Топ-20
top_indices = np.argsort(importances)[::-1][:20]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(top_indices)), importances[top_indices])
ax.set_yticks(range(len(top_indices)))
ax.set_yticklabels([feature_names[i] for i in top_indices])
ax.invert_yaxis()
ax.set_xlabel('Важность')
ax.set_title('Топ-20 признаков (CatBoost Tweedie)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Контрпример: известные нули

**Постановка:** магазин закрыт по понедельникам → нули известны заранее.
Croston прогнозирует ненулевое значение; глобальная модель с признаком `is_closed` даёт точный 0.

In [ ]:
# Загружаем синтетические данные
df_known_zeros = pd.read_csv('../data/intermittent_demand.csv')
df_known_zeros['date'] = pd.to_datetime(df_known_zeros['date'])

print(f"Контрпример: {len(df_known_zeros)} строк, период {df_known_zeros['date'].min()} — {df_known_zeros['date'].max()}")

In [ ]:
# EDA
df_known_zeros['day_of_week'] = df_known_zeros['date'].dt.dayofweek

zero_by_day = df_known_zeros.groupby('day_of_week')['sales'].apply(lambda x: (x == 0).mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Временной ряд
axes[0].plot(df_known_zeros['date'], df_known_zeros['sales'], linewidth=0.8)
axes[0].set_ylabel('Продажи')
axes[0].set_title('Ряд с известными нулями')
axes[0].grid(alpha=0.3)

# Доля нулей по дням недели
axes[1].bar(range(7), zero_by_day.values)
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс'])
axes[1].set_ylabel('Доля нулей')
axes[1].set_title('Доля нулей по дням недели')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nДоля нулей по понедельникам: {zero_by_day[0]:.2f}")

In [ ]:
# Train/test split
H_kz = 28
train_kz = df_known_zeros.iloc[:-H_kz].copy()
test_kz = df_known_zeros.iloc[-H_kz:].copy()

# Признаки
train_kz['is_closed'] = (train_kz['day_of_week'] == 0).astype(int)
test_kz['is_closed'] = (test_kz['day_of_week'] == 0).astype(int)

# Лаги
for lag in [H_kz, H_kz+7]:
    train_kz[f'lag_{lag}'] = train_kz['sales'].shift(lag)
    test_kz[f'lag_{lag}'] = test_kz['sales'].shift(lag)

train_kz = train_kz.dropna()

X_train_kz = train_kz[['is_closed', f'lag_{H_kz}', f'lag_{H_kz+7}']]
y_train_kz = train_kz['sales']

X_test_kz = test_kz[['is_closed', f'lag_{H_kz}', f'lag_{H_kz+7}']]
y_test_kz = test_kz['sales']

In [ ]:
# Модель с признаком is_closed
cb_kz = CatBoostRegressor(
    iterations=200,
    depth=4,
    learning_rate=0.1,
    loss_function='Tweedie:variance_power=1.5',
    verbose=False,
    random_state=42
)

cb_kz.fit(X_train_kz, y_train_kz)
preds_kz = cb_kz.predict(X_test_kz)
preds_kz = np.maximum(preds_kz, 0)

In [ ]:
# Croston для сравнения
train_sf_kz = pd.DataFrame({
    'unique_id': 'item_1',
    'ds': train_kz['date'],
    'y': train_kz['sales']
})

sf_kz = StatsForecast(models=[CrostonSBA()], freq='D', n_jobs=1)
sf_kz.fit(train_sf_kz)
forecast_kz = sf_kz.predict(h=H_kz)['CrostonSBA'].values

In [ ]:
# Сравнение
plt.figure(figsize=(14, 4))
plt.plot(range(H_kz), y_test_kz.values, 'o-', label='Факт', markersize=4)
plt.plot(range(H_kz), preds_kz, 's-', label='CatBoost (с is_closed)', markersize=4)
plt.plot(range(H_kz), forecast_kz, '^-', label='CrostonSBA', markersize=4)
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.legend()
plt.title('Контрпример: известные нули')
plt.xlabel('День')
plt.ylabel('Продажи')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Метрики с маской (исключаем известные нули)
mask_open = (test_kz['is_closed'] == 0).values

rmse_cb_masked = np.sqrt(((y_test_kz[mask_open].values - preds_kz[mask_open]) ** 2).mean())
rmse_croston_masked = np.sqrt(((y_test_kz[mask_open].values - forecast_kz[mask_open]) ** 2).mean())

rmse_cb_all = np.sqrt(((y_test_kz.values - preds_kz) ** 2).mean())
rmse_croston_all = np.sqrt(((y_test_kz.values - forecast_kz) ** 2).mean())

print(f"\nRMSE (только открытые дни):")
print(f"  CatBoost:   {rmse_cb_masked:.3f}")
print(f"  CrostonSBA: {rmse_croston_masked:.3f}")

print(f"\nRMSE (все дни):")
print(f"  CatBoost:   {rmse_cb_all:.3f}")
print(f"  CrostonSBA: {rmse_croston_all:.3f}")

print("\n→ CatBoost точно прогнозирует 0 в закрытые дни, Croston — нет.")

## 7. Итоговая памятка выбора подхода

| Ситуация | Рекомендуемый метод | Почему |
|----------|-------------------|--------|
| **Intermittent, мало признаков** | CrostonSBA / TSB | Специализированы для нулей |
| **Lumpy, нестабильность** | ADIDA | Агрегация снижает шум |
| **Много признаков (календарь, категории)** | CatBoost Tweedie | Учитывает контекст |
| **Overdispersion (var > mean)** | Tweedie, не Poisson | Poisson предполагает равнодисперсность |
| **Известные нули (закрытие, сезон)** | Глобальная модель с признаком | Croston их не учитывает |
| **Короткая история (<50 точек)** | Croston/TSB | Бустинг переобучается |
| **Нужны интервалы** | Conformal + любая модель | Croston даёт только точечный прогноз |

### Практические рекомендации

1. **Начните с классификации ADI/CV²** (10.B.1) → понять природу спроса.
2. **Если мало признаков:**
   - Intermittent → CrostonSBA
   - Lumpy → ADIDA
3. **Если много признаков** (календарь, иерархия, внешние данные):
   - CatBoost Tweedie с `cat_features`
   - Лаги ≥ H, скользящие окна со сдвигом H
4. **Tweedie лучше Poisson** для прерывистых данных (variance_power=1.5).
5. **Обрезайте отрицательные** прогнозы: `np.maximum(pred, 0)`.
6. **Валидация:** честная кросс-валидация с перестроением признаков на каждом фолде.
7. **Метрики:** RMSSE + bias, не MAE (занижает нулевой прогноз).

### Финальное сравнение на M5 CA_1

- **Smooth:** CatBoost RMSE / Tweedie ≈ 10-15% лучше Croston
- **Intermittent:** CatBoost Tweedie ≈ CrostonSBA / TSB
- **Erratic:** CatBoost Tweedie ≈ 5-10% лучше
- **Lumpy:** ADIDA ≈ CatBoost Tweedie, оба лучше RMSE

**Вывод:** глобальная модель с Tweedie — хороший универсальный выбор, но для чисто прерывистых рядов без дополнительных признаков Croston/ADIDA конкурентны.